# Task 3: Real-Time Echocardiogram Video Analysis

Real-time OpenCV pipeline on ultrasound .mp4 video.
Applies histogram EQ, COLORMAP_JET, color balance, log transform, and gamma
to each frame, then displays raw vs enhanced side-by-side.


In [1]:
import cv2
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import os

os.makedirs('output', exist_ok=True)


## Helper: Per-Frame Enhancement Pipeline


In [2]:
def enhance_frame(frame_bgr):
    # step 1: grayscale -- ultrasound is single-channel anyway
    gray = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2GRAY)
    # step 2: histogram equalization -- combat murky contrast
    eq = cv2.equalizeHist(gray)
    # step 3: COLORMAP_JET -- highlight blood flow intensities
    heatmap = cv2.applyColorMap(eq, cv2.COLORMAP_JET)
    # step 4: gray-world color balance -- neutralize colormap cast
    bal = heatmap.astype('float32')
    gm = bal.mean()
    for ch in range(3):
        cm = bal[:,:,ch].mean()
        if cm > 0:
            bal[:,:,ch] *= gm / cm
    bal = bal.clip(0, 255).astype('uint8')
    # step 5: log transform -- reveal dark heart chambers
    lc = 255.0 / np.log1p(255.0)
    lf = (lc * np.log1p(bal.astype('float32'))).clip(0, 255).astype('uint8')
    # step 6: gamma < 1 -- suppress white backscatter probe noise
    enhanced = (np.power(lf.astype('float32') / 255.0, 0.6) * 255).clip(0, 255).astype('uint8')
    return enhanced


## Step 1: Video Capture Setup (cv2.VideoCapture)


In [3]:
VIDEO_PATH = 'data/sample_echo.mp4'
cap = cv2.VideoCapture(VIDEO_PATH)
if not cap.isOpened():
    raise IOError(f'Cannot open video: {VIDEO_PATH}')

fps   = cap.get(cv2.CAP_PROP_FPS)
w     = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h     = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
cap.release()

print(f'Video loaded: {w}x{h}, {fps:.1f} FPS, {total} frames')


Video loaded: 112x112, 50.0 FPS, 174 frames


## Step 2: Real-Time Processing Loop

Runs a `while` loop on each frame. Tries `cv2.imshow()` (interactive),
falls back gracefully to headless mode (saving PNGs) if no display is available.


In [4]:
cap = cv2.VideoCapture('data/sample_echo.mp4')
frame_count = 0
sample_frames = []
SAVE_EVERY = 30
has_display = True

print('Starting pipeline loop...')

while True:
    ret, frame = cap.read()
    if not ret:
        break  # end of video

    frame_count += 1
    enhanced = enhance_frame(frame)

    if frame.shape != enhanced.shape:
        enhanced = cv2.resize(enhanced, (frame.shape[1], frame.shape[0]))

    # build monitoring array: raw | enhanced
    monitor = np.hstack([frame, enhanced])
    cv2.putText(monitor, 'RAW ULTRASOUND',
                (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255,255,255), 2)
    cv2.putText(monitor, 'ENHANCED PIPELINE',
                (frame.shape[1]+10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,255,128), 2)
    cv2.putText(monitor, f'Frame: {frame_count}',
                (10, monitor.shape[0]-10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (200,200,200), 1)

    if has_display:
        try:
            cv2.imshow('Echocardiogram Analysis', monitor)
            key = cv2.waitKey(1) & 0xFF
            if key == ord('q'):
                print('Q pressed -- stopping.')
                break
        except cv2.error:
            has_display = False
            print('No display -- headless mode, saving frames instead.')

    if frame_count % SAVE_EVERY == 0 or frame_count == 1:
        sample_frames.append((frame_count, monitor.copy()))

cap.release()
cv2.destroyAllWindows()
print(f'Done. Processed {frame_count} frames.')


Starting pipeline loop...


Done. Processed 174 frames.


## Step 3: Save Monitoring Array Screenshots


In [5]:
for fidx, arr in sample_frames:
    fname = f'output/frame_sample_{fidx:04d}.png'
    cv2.imwrite(fname, arr)
    print(f'Saved -> {fname}')

if sample_frames:
    first_fidx, first_arr = sample_frames[0]
    plt.figure(figsize=(16, 6))
    plt.imshow(cv2.cvtColor(first_arr, cv2.COLOR_BGR2RGB))
    plt.title('Monitoring Array: Raw Ultrasound (left) | Enhanced Pipeline (right)', fontsize=13)
    plt.axis('off')
    plt.tight_layout()
    plt.savefig('output/pipeline_screenshot_example.png', dpi=150)
    plt.show()
    print('Saved -> output/pipeline_screenshot_example.png')


Saved -> output/frame_sample_0001.png
Saved -> output/frame_sample_0030.png
Saved -> output/frame_sample_0060.png
Saved -> output/frame_sample_0090.png
Saved -> output/frame_sample_0120.png
Saved -> output/frame_sample_0150.png


Saved -> output/pipeline_screenshot_example.png


C:\Users\hp z book\AppData\Local\Temp\ipykernel_2920\1920375004.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Step 4: Static Multi-Frame Preview (4 key frames)


In [6]:
cap = cv2.VideoCapture('data/sample_echo.mp4')
total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
indices = [int(total * f) for f in [0.1, 0.3, 0.6, 0.85]]
previews = []
for idx in indices:
    cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
    ret, frame = cap.read()
    if ret:
        previews.append((frame, enhance_frame(frame), idx))
cap.release()

fig, axes = plt.subplots(len(previews), 2, figsize=(14, 5*len(previews)))
fig.suptitle('Task 3: Real-Time Echo Analysis -- Raw vs Enhanced', fontsize=14, fontweight='bold')
for row, (raw, enh, fidx) in enumerate(previews):
    axes[row,0].imshow(cv2.cvtColor(raw, cv2.COLOR_BGR2RGB))
    axes[row,0].set_title(f'Frame {fidx} -- Raw Ultrasound'); axes[row,0].axis('off')
    axes[row,1].imshow(cv2.cvtColor(enh, cv2.COLOR_BGR2RGB))
    axes[row,1].set_title(f'Frame {fidx} -- Enhanced Pipeline'); axes[row,1].axis('off')

plt.tight_layout()
plt.savefig('output/static_preview_frames.png', dpi=150)
plt.show()
print('Saved -> output/static_preview_frames.png')
print('All done!')


Saved -> output/static_preview_frames.png
All done!


C:\Users\hp z book\AppData\Local\Temp\ipykernel_2920\3160814983.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
